# WWI 01 · Bronze → Silver
Adapted from Microsoft's `01 - Create Delta Tables` for the LAB sandbox.

- **Default lakehouse:** `lh_bronze` (raw files in `Files/wwi-raw-data`)
- **Also attached:** `lh_silver` (Delta tables are written here)


### 1 · Spark session config
V-Order speeds up Direct Lake and SQL endpoint reads. Optimize Write produces fewer, larger files.

In [ ]:
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")

### 2 · Fact · Sale → `lh_silver.fact_sale`
Reads raw parquet from bronze, derives Year/Quarter/Month, and writes a Delta table partitioned by Year/Quarter into silver.

In [ ]:
from pyspark.sql.functions import col, year, month, quarter, current_timestamp, lit

RAW = "Files/wwi-raw-data/full/"
SILVER = "lh_silver"

df = spark.read.format("parquet").load(RAW + "fact_sale_1y_full")
df = (df.withColumn("Year", year(col("InvoiceDateKey")))
        .withColumn("Quarter", quarter(col("InvoiceDateKey")))
        .withColumn("Month", month(col("InvoiceDateKey")))
        .withColumn("SourceSystem", lit("WWI"))
        .withColumn("LoadedAt", current_timestamp()))

(df.write.mode("overwrite").format("delta")
   .option("overwriteSchema", "true")
   .partitionBy("Year", "Quarter")
   .saveAsTable(f"{SILVER}.fact_sale"))

### 3 · Dimensions → `lh_silver`
One reusable function, looped over five dimensions. This is the seed of a metadata-driven loader.

In [ ]:
def load_full(table_name):
    df = spark.read.format("parquet").load(RAW + table_name)
    if "Photo" in df.columns:
        df = df.drop("Photo")
    df = df.withColumn("SourceSystem", lit("WWI")).withColumn("LoadedAt", current_timestamp())
    df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.{table_name}")
    print(f"{table_name:<22} {df.count():>10,} rows")

for t in ["dimension_city", "dimension_customer", "dimension_date", "dimension_employee", "dimension_stock_item"]:
    load_full(t)

### 4 · Check your work

In [ ]:
display(spark.sql(f"SELECT Year, Quarter, COUNT(*) AS Rows FROM {SILVER}.fact_sale GROUP BY Year, Quarter ORDER BY Year, Quarter"))

In [ ]:
%%sql
DESCRIBE HISTORY lh_silver.fact_sale